# 01 - Dataset prep (LettuceMOTS -> YOLO detection, single class)

**Goal:** verify the label format, split train/val **by sequence folder**, convert polygon labels to axis-aligned boxes, and emit the Ultralytics data yaml (`nc=1`, `names=['lettuce']`).

The LettuceMOTS ground truth is YOLOv5 **segmentation polygons**. We derive one box per polygon from its min/max normalized coordinates - deterministic, straight from the real annotations. Single class only; no weed/disease classes are added.

Split is **by sequence folder**: frames within a sequence are consecutive video frames, so a whole sequence goes entirely to train or entirely to val. Random per-frame splitting would leak near-identical adjacent frames across the split.

In [1]:
import os, sys
from pathlib import Path

# Locate repo root (holds croprow/utils.py) so `import croprow.utils` works
# regardless of the cwd the notebook is launched from.
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "croprow" / "utils.py").is_file()
)
sys.path.insert(0, str(REPO_ROOT))
from croprow import utils as U

# === CONFIG: the ONLY place to set the dataset location ===================
# Prefer the LETTUCE_ROOT env var. Otherwise edit LETTUCE_ROOT_DEFAULT below.
# (No absolute path is baked into utils.py -- it is supplied from here.)
LETTUCE_ROOT_DEFAULT = r"D:\croprow_dataset\LettuceMOTS"
LETTUCE_ROOT = os.environ.get("LETTUCE_ROOT", LETTUCE_ROOT_DEFAULT)
# ==========================================================================

root = U.resolve_lettuce_root(LETTUCE_ROOT)
DATA_DIR = REPO_ROOT / "croprow" / "data"
RESULTS_MD = REPO_ROOT / "croprow" / "RESULTS.md"
print("LETTUCE_ROOT :", root)
print("DATA_DIR     :", DATA_DIR)

# Split / conversion parameters
VAL_FRAC = 0.25
SEED = 42

LETTUCE_ROOT : D:\croprow_dataset\LettuceMOTS
DATA_DIR     : D:\sih26software\Crop_detection_management\croprow\data


## 1. Verify label format

A detection box line is exactly 5 tokens (`class cx cy w h`); a segmentation polygon line is an odd count > 5.

In [2]:
fmt = U.inspect_label_format(root)
for k, v in fmt.items():
    print(f"  {k:14s}: {v}")

if fmt["is_boxes"]:
    print("\nLabels are already YOLO boxes -- conversion is a no-op passthrough.")
elif fmt["is_polygons"]:
    print("\nLabels are segmentation polygons -> will convert to axis-aligned boxes.")
else:
    raise RuntimeError(
        f"Unexpected label format (box={fmt['box_lines']}, "
        f"poly={fmt['polygon_lines']}, other={fmt['other_lines']}). "
        "Assumption failed -- stop and inspect the dataset."
    )

  files_scanned : 596
  total_lines   : 7844
  box_lines     : 0
  polygon_lines : 7844
  other_lines   : 0
  min_tokens    : 9
  max_tokens    : 347
  is_boxes      : False
  is_polygons   : True

Labels are segmentation polygons -> will convert to axis-aligned boxes.


## 2. Sequences and split (by folder)

In [3]:
seqs = U.labeled_sequences(root)
print("labeled train sequences:", seqs)
print("test sequences (no labels, not used for training):", U.test_sequences(root))

train_seqs, val_seqs = U.split_sequences(seqs, val_frac=VAL_FRAC, seed=SEED)
print("\nTRAIN sequences:", train_seqs)
print("VAL   sequences:", val_seqs)
assert not (set(train_seqs) & set(val_seqs)), "sequence leaked across split!"

labeled train sequences: ['0000', '0001', '0002', '0004', '0005', '0006', '0008', '0009', '0010']
test sequences (no labels, not used for training): ['0003', '0007', '0011']

TRAIN sequences: ['0000', '0001', '0002', '0005', '0006', '0009', '0010']
VAL   sequences: ['0004', '0008']


## 3. Convert polygons -> boxes

Written to `<LETTUCE_ROOT>/train/labels/<seq>/<frame>.txt` (the YOLO-standard mirror of `train/images`, so Ultralytics finds them automatically).

In [4]:
per_seq = U.convert_all(root, seqs)
for s in seqs:
    print(f"  seq {s}: {per_seq[s]:6d} boxes")
print("  total boxes:", sum(per_seq.values()))

  seq 0000:   1243 boxes
  seq 0001:    678 boxes
  seq 0002:    571 boxes
  seq 0004:   1444 boxes
  seq 0005:    669 boxes
  seq 0006:    577 boxes
  seq 0008:   1252 boxes
  seq 0009:    733 boxes
  seq 0010:    677 boxes
  total boxes: 7844


## 4. Write train.txt / val.txt and the data yaml

In [5]:
train_imgs = U.image_paths_for_seqs(root, train_seqs)
val_imgs   = U.image_paths_for_seqs(root, val_seqs)
n_tr = U.write_list_file(DATA_DIR / "train.txt", train_imgs)
n_va = U.write_list_file(DATA_DIR / "val.txt", val_imgs)
print(f"train.txt: {n_tr} images\nval.txt  : {n_va} images")

yaml_path = U.make_data_yaml(DATA_DIR / "lettuce.yaml",
                             DATA_DIR / "train.txt", DATA_DIR / "val.txt")
print("\nwrote", yaml_path, "\n")
print(yaml_path.read_text())

train.txt: 383 images
val.txt  : 213 images

wrote D:\sih26software\Crop_detection_management\croprow\data\lettuce.yaml 

train: D:\sih26software\Crop_detection_management\croprow\data\train.txt
val: D:\sih26software\Crop_detection_management\croprow\data\val.txt
nc: 1
names:
- lettuce



## 5. Per-split summary (images + boxes)

In [6]:
def summarize(name, split_seqs):
    imgs = U.image_paths_for_seqs(root, split_seqs)
    boxes = U.count_boxes_for_seqs(root, split_seqs)
    print(f"{name:5s} | seqs={len(split_seqs):2d} | images={len(imgs):5d} | boxes={boxes:6d}")
    return len(imgs), boxes

print("split | #seqs | #images | #boxes")
print("-" * 44)
tr = summarize("train", train_seqs)
va = summarize("val", val_seqs)
print("-" * 44)
print(f"TOTAL | seqs={len(seqs):2d} | images={tr[0]+va[0]:5d} | boxes={tr[1]+va[1]:6d}")

split | #seqs | #images | #boxes
--------------------------------------------


train | seqs= 7 | images=  383 | boxes=  5148


val   | seqs= 2 | images=  213 | boxes=  2696
--------------------------------------------
TOTAL | seqs= 9 | images=  596 | boxes=  7844
